# Here's the plan chalked out before building anything.

**What we want to build:** A personal financial portfolio dashboard on Kaggle, powered by OpenClaw (open-source LLM + agent tooling), that gives you a unified view across Indian investment instruments — MF, Equity, FD, NPS, PPF, etc. — with intelligent analysis.

Let me diagram the full architecture first:Click any box to deep-dive into that component. Here's the full plan in plain language:

**What we want to accomplish**

Build a personal financial intelligence system on Kaggle that acts as your AI-powered portfolio advisor — you ask it questions in natural language and it answers from your actual data.

**The five layers**

The data layer ingests all your instruments: MF (NAV history via AMFI or MFAPI), equity (NSE/BSE via `yfinance` or Zerodha Kite), FD/RD (manual), NPS (PFRDA CAS PDF), PPF (manual or portal export), bonds, and gold ETFs.

The Kaggle ETL layer (Python notebooks + Kaggle Datasets) cleans and standardises everything — computes XIRR for each instrument, normalises dates, and stores in a lightweight SQLite or a versioned CSV dataset.

The OpenClaw agent layer is where the AI lives. OpenClaw (being open-source, it runs on free Kaggle P100/T4 GPUs) orchestrates a local LLM (Mistral-7B or LLaMA-3 via Hugging Face), a RAG pipeline over your portfolio data (ChromaDB or FAISS as vector store), and custom tools — a tax calculator (LTCG/STCG rules, 80C deductions), a returns comparator, and a rebalancing advisor.

The analysis layer is what you can actually query: "What is my total net worth today?", "Which instrument gave the best CAGR over 5 years?", "How much tax will I pay if I redeem this MF?", "Am I on track for a ₹5Cr retirement corpus at 60?"

The output layer renders everything in a Gradio chat UI (hostable on Kaggle), Plotly dashboards, and auto-generated markdown/PDF reports — and Kaggle's scheduled notebook feature runs it weekly so data stays fresh.

**What makes Kaggle ideal for this**

Free GPU access for the LLM inference, persistent datasets for your portfolio data, Kaggle Secrets for any API keys (MFAPI, Zerodha), and no infra management. OpenClaw fits cleanly because it's designed for exactly this kind of agentic, tool-using, RAG-backed workflow on commodity hardware.

**Next steps — to start building?**

(1) the full data ingestion notebook for all instrument types, (2) the OpenClaw agent setup with RAG + tools, or (3) the Gradio UI shell. Which piece do you want first?

# 📊 Personal Financial Portfolio — Data Ingestion Notebook
**Instruments covered:** Mutual Funds · Equity · FD/RD · NPS · PPF · Gold ETF / Bonds

**How to use:**
1. Upload your instrument CSV files to `/kaggle/input/my-portfolio/`
2. Run all cells top-to-bottom
3. Final unified portfolio DataFrame is saved to `/kaggle/working/portfolio_master.csv`
4. Summary stats are printed at the end

**CSV templates** are generated in Cell 1 — download and fill them, then re-upload.

In [1]:
# ── Cell 0: Install dependencies ──────────────────────────────────────────────
!pip install -q yfinance mftool pandas numpy requests scipy openpyxl pyxirr

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 7.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 533.1/533.1 kB 30.9 MB/s eta 0:00:00


In [2]:
# ── Cell 1: Imports & config ───────────────────────────────────────────────────
import os, json, warnings
import pandas as pd
import numpy as np
import requests
import yfinance as yf
from datetime import date, datetime, timedelta
from pyxirr import xirr
from mftool import Mftool
mf = Mftool()
quote = mf.get_scheme_quote('119597')
print(quote)
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────────────────────────────────
INPUT_DIR  = '/kaggle/input/my-portfolio'   # Upload your CSVs here
OUTPUT_DIR = '/kaggle/working'
os.makedirs(OUTPUT_DIR, exist_ok=True)

TODAY = date.today()
print(f'Portfolio ingestion started: {TODAY}')

{'scheme_code': '119597', 'scheme_name': 'Sundaram Financial Services Opportunities Fund Direct Plan - Growth', 'last_updated': '09-Apr-2026', 'nav': '117.5463'}
Portfolio ingestion started: 2026-04-10


In [3]:
# ── Cell 2: Generate CSV templates (run once, download, fill, re-upload) ──────
templates = {
    'mf_holdings.csv': [
        {'scheme_name': 'Axis Bluechip Fund Direct Growth',
         'amfi_code': '120465',
         'units': 250.567,
         'avg_nav': 42.15,
         'invested_amount': 10000,
         'purchase_date': '2022-04-01',
         'folio_number': 'XXXX1234',
         'sip_monthly': 2000}
    ],
    'mf_transactions.csv': [
        {'scheme_name': 'Axis Bluechip Fund Direct Growth',
         'amfi_code': '120465',
         'txn_date': '2022-04-01',
         'txn_type': 'BUY',        # BUY / SELL / DIVIDEND
         'units': 50.00,
         'nav': 40.00,
         'amount': 2000}
    ],
    'equity_holdings.csv': [
        {'ticker': 'RELIANCE.NS',
         'company_name': 'Reliance Industries',
         'quantity': 10,
         'avg_buy_price': 2450.00,
         'purchase_date': '2021-06-15',
         'exchange': 'NSE',
         'segment': 'EQUITY'}      # EQUITY / ETF
    ],
    'equity_transactions.csv': [
        {'ticker': 'RELIANCE.NS',
         'txn_date': '2021-06-15',
         'txn_type': 'BUY',        # BUY / SELL
         'quantity': 10,
         'price': 2450.00,
         'brokerage': 20.00,
         'stt': 12.25}
    ],
    'fd_rd_holdings.csv': [
        {'bank_name': 'HDFC Bank',
         'type': 'FD',             # FD / RD
         'principal': 100000,
         'annual_rate_pct': 7.25,
         'start_date': '2023-01-10',
         'maturity_date': '2025-01-10',
         'compounding': 'Quarterly', # Quarterly / Monthly / Annual
         'account_number': 'FD00XXXX',
         'is_tax_saver': False}
    ],
    'nps_holdings.csv': [
        {'pran': 'XXXXXXXXXX',
         'tier': 'Tier-I',          # Tier-I / Tier-II
         'fund_manager': 'SBI Pension Funds',
         'scheme': 'Scheme E - Tier I',
         'units': 1200.00,
         'nav': 45.00,
         'total_invested': 50000,
         'as_of_date': '2024-03-31'}
    ],
    'ppf_holdings.csv': [
        {'bank_name': 'SBI',
         'account_number': 'PPFXXXXXXXX',
         'opening_date': '2015-04-01',
         'maturity_date': '2030-04-01',
         'current_balance': 350000,
         'annual_deposit': 150000,
         'interest_rate_pct': 7.1,
         'last_updated': '2024-03-31'}
    ],
    'gold_bonds_holdings.csv': [
        {'instrument_name': 'Sovereign Gold Bond 2023-24 Series I',
         'type': 'SGB',            # SGB / GOLD_ETF / GOVT_BOND / CORP_BOND
         'ticker': 'SGBAUG28.NS',  # for ETFs / listed bonds
         'units_or_grams': 10,
         'issue_price': 5900,
         'purchase_date': '2023-08-18',
         'maturity_date': '2031-08-18',
         'interest_rate_pct': 2.5,  # SGB annual coupon
         'face_value': 1000}       # for bonds
    ]
}

for fname, rows in templates.items():
    path = os.path.join(OUTPUT_DIR, f'TEMPLATE_{fname}')
    pd.DataFrame(rows).to_csv(path, index=False)
    print(f'✅ Template saved: TEMPLATE_{fname}')

print('\nDownload templates from /kaggle/working/, fill with your data, re-upload to /kaggle/input/my-portfolio/')

✅ Template saved: TEMPLATE_mf_holdings.csv
✅ Template saved: TEMPLATE_mf_transactions.csv
✅ Template saved: TEMPLATE_equity_holdings.csv
✅ Template saved: TEMPLATE_equity_transactions.csv
✅ Template saved: TEMPLATE_fd_rd_holdings.csv
✅ Template saved: TEMPLATE_nps_holdings.csv
✅ Template saved: TEMPLATE_ppf_holdings.csv
✅ Template saved: TEMPLATE_gold_bonds_holdings.csv

Download templates from /kaggle/working/, fill with your data, re-upload to /kaggle/input/my-portfolio/


In [4]:
# ── Cell 3: Helper utilities ───────────────────────────────────────────────────

def safe_read(filename, required_cols=None):
    """Read a CSV from INPUT_DIR; return empty DF if not found."""
    path = os.path.join(INPUT_DIR, filename)
    if not os.path.exists(path):
        print(f'  ⚠️  {filename} not found — skipping')
        return pd.DataFrame()
    df = pd.read_csv(path)
    if required_cols:
        missing = [c for c in required_cols if c not in df.columns]
        if missing:
            raise ValueError(f'{filename} missing columns: {missing}')
    print(f'  ✅ {filename}: {len(df)} rows loaded')
    return df


def compute_xirr(dates, amounts):
    """XIRR wrapper — negative for investments, positive for current value."""
    try:
        return round(xirr(dates, amounts) * 100, 2)
    except Exception:
        return None


def cagr(invested, current, years):
    """Compound Annual Growth Rate."""
    if invested <= 0 or years <= 0:
        return None
    return round(((current / invested) ** (1 / years) - 1) * 100, 2)


def fd_maturity_value(principal, rate_pct, years, compounding='Quarterly'):
    """Compute FD maturity value with compound interest."""
    n_map = {'Annual': 1, 'Half-Yearly': 2, 'Quarterly': 4, 'Monthly': 12}
    n = n_map.get(compounding, 4)
    r = rate_pct / 100
    return round(principal * (1 + r / n) ** (n * years), 2)


def ppf_projected_balance(current_balance, annual_deposit, rate_pct, years_left):
    """Project PPF balance — end-of-year deposits, annual compounding."""
    r = rate_pct / 100
    bal = current_balance
    for _ in range(years_left):
        bal = (bal + annual_deposit) * (1 + r)
    return round(bal, 2)


def gold_spot_price_inr():
    """Fetch approximate gold price in INR per gram via Yahoo Finance (GC=F * USDINR)."""
    try:
        gc  = yf.Ticker('GC=F').history(period='1d')['Close'].iloc[-1]
        fx  = yf.Ticker('USDINR=X').history(period='1d')['Close'].iloc[-1]
        per_gram = (gc * fx) / 31.1035  # troy oz -> gram
        return round(per_gram, 2)
    except Exception:
        return None

print('Helpers loaded ✅')

Helpers loaded ✅


In [5]:
# ── Cell 4: MUTUAL FUNDS ───────────────────────────────────────────────────────
print('=== Mutual Funds ===')

mf = safe_read('mf_holdings.csv')
mf_txn = safe_read('mf_transactions.csv')

mf_records = []

if not mf.empty:
    for _, row in mf.iterrows():
        amfi_code = str(int(row['amfi_code']))
        # Fetch current NAV from MFAPI (free, no auth)
        try:
            resp = requests.get(
                f'https://api.mfapi.in/mf/{amfi_code}', timeout=10
            ).json()
            current_nav = float(resp['data'][0]['nav'])
            scheme_name = resp['meta']['scheme_name']
        except Exception:
            current_nav = row['avg_nav']   # fallback to cost NAV
            scheme_name = row['scheme_name']

        units = float(row['units'])
        invested = float(row['invested_amount'])
        current_value = round(units * current_nav, 2)
        gain = round(current_value - invested, 2)
        gain_pct = round((gain / invested) * 100, 2) if invested else 0
        purchase_date = pd.to_datetime(row['purchase_date']).date()
        years_held = (TODAY - purchase_date).days / 365.25
        cagr_val = cagr(invested, current_value, years_held)

        # LTCG / STCG classification (equity MF: 1yr, debt MF: 3yr — simplified)
        tax_type = 'LTCG' if years_held >= 1 else 'STCG'

        mf_records.append({
            'instrument_type': 'Mutual Fund',
            'name': scheme_name,
            'invested_amount': invested,
            'current_value': current_value,
            'gain_loss': gain,
            'gain_loss_pct': gain_pct,
            'cagr_pct': cagr_val,
            'purchase_date': str(purchase_date),
            'years_held': round(years_held, 2),
            'tax_type': tax_type,
            'notes': f'Units: {units} | NAV: ₹{current_nav}'
        })

mf_df = pd.DataFrame(mf_records)
if not mf_df.empty:
    print(f'  Mutual Funds total invested : ₹{mf_df["invested_amount"].sum():,.0f}')
    print(f'  Mutual Funds current value  : ₹{mf_df["current_value"].sum():,.0f}')
print()

=== Mutual Funds ===
  ⚠️  mf_holdings.csv not found — skipping
  ⚠️  mf_transactions.csv not found — skipping



In [6]:
# ── Cell 5: EQUITY (NSE / BSE stocks) ─────────────────────────────────────────
print('=== Equity ===')

eq = safe_read('equity_holdings.csv')
eq_records = []

if not eq.empty:
    tickers = eq['ticker'].tolist()
    # Batch fetch current prices
    price_map = {}
    for ticker in tickers:
        try:
            hist = yf.Ticker(ticker).history(period='1d')
            price_map[ticker] = round(hist['Close'].iloc[-1], 2)
        except Exception:
            price_map[ticker] = None
            print(f'  ⚠️  Could not fetch price for {ticker}')

    for _, row in eq.iterrows():
        ticker = row['ticker']
        qty = float(row['quantity'])
        avg_price = float(row['avg_buy_price'])
        invested = round(qty * avg_price, 2)
        current_price = price_map.get(ticker) or avg_price
        current_value = round(qty * current_price, 2)
        gain = round(current_value - invested, 2)
        gain_pct = round((gain / invested) * 100, 2) if invested else 0
        purchase_date = pd.to_datetime(row['purchase_date']).date()
        years_held = (TODAY - purchase_date).days / 365.25
        cagr_val = cagr(invested, current_value, years_held)
        tax_type = 'LTCG' if years_held >= 1 else 'STCG'

        eq_records.append({
            'instrument_type': 'Equity',
            'name': row.get('company_name', ticker),
            'invested_amount': invested,
            'current_value': current_value,
            'gain_loss': gain,
            'gain_loss_pct': gain_pct,
            'cagr_pct': cagr_val,
            'purchase_date': str(purchase_date),
            'years_held': round(years_held, 2),
            'tax_type': tax_type,
            'notes': f'Qty: {qty} | LTP: ₹{current_price}'
        })

eq_df = pd.DataFrame(eq_records)
if not eq_df.empty:
    print(f'  Equity total invested : ₹{eq_df["invested_amount"].sum():,.0f}')
    print(f'  Equity current value  : ₹{eq_df["current_value"].sum():,.0f}')
print()

=== Equity ===
  ⚠️  equity_holdings.csv not found — skipping



In [7]:
# ── Cell 6: FIXED DEPOSITS & RECURRING DEPOSITS ────────────────────────────────
print('=== FD / RD ===')

fd = safe_read('fd_rd_holdings.csv')
fd_records = []

if not fd.empty:
    for _, row in fd.iterrows():
        principal = float(row['principal'])
        rate = float(row['annual_rate_pct'])
        start = pd.to_datetime(row['start_date']).date()
        maturity = pd.to_datetime(row['maturity_date']).date()
        total_years = (maturity - start).days / 365.25
        elapsed_years = max((TODAY - start).days / 365.25, 0)
        compounding = row.get('compounding', 'Quarterly')

        maturity_value = fd_maturity_value(principal, rate, total_years, compounding)
        # Accrued value today
        accrued = fd_maturity_value(principal, rate, min(elapsed_years, total_years), compounding)
        interest_earned = round(accrued - principal, 2)

        status = 'Matured' if TODAY >= maturity else 'Active'
        days_left = max((maturity - TODAY).days, 0)

        fd_records.append({
            'instrument_type': row['type'],
            'name': f"{row['bank_name']} {row['type']} @ {rate}%",
            'invested_amount': principal,
            'current_value': accrued,
            'gain_loss': interest_earned,
            'gain_loss_pct': round((interest_earned / principal) * 100, 2),
            'cagr_pct': rate,   # FD rate IS the guaranteed return
            'purchase_date': str(start),
            'years_held': round(elapsed_years, 2),
            'tax_type': 'Interest (slab)',
            'notes': f'Maturity: {maturity} | Value at maturity: ₹{maturity_value:,.0f} | Status: {status} | Days left: {days_left}'
        })

fd_df = pd.DataFrame(fd_records)
if not fd_df.empty:
    print(f'  FD/RD total principal : ₹{fd_df["invested_amount"].sum():,.0f}')
    print(f'  FD/RD accrued value   : ₹{fd_df["current_value"].sum():,.0f}')
print()

=== FD / RD ===
  ⚠️  fd_rd_holdings.csv not found — skipping



In [8]:
# ── Cell 7: NPS ────────────────────────────────────────────────────────────────
print('=== NPS ===')

nps = safe_read('nps_holdings.csv')
nps_records = []

if not nps.empty:
    for _, row in nps.iterrows():
        units = float(row['units'])
        nav = float(row['nav'])
        invested = float(row['total_invested'])
        current_value = round(units * nav, 2)
        gain = round(current_value - invested, 2)
        gain_pct = round((gain / invested) * 100, 2) if invested else 0
        as_of = pd.to_datetime(row['as_of_date']).date()
        years_held = (as_of - date(2004, 1, 1)).days / 365.25  # NPS launched 2004

        nps_records.append({
            'instrument_type': 'NPS',
            'name': f"{row['fund_manager']} — {row['scheme']} ({row['tier']})",
            'invested_amount': invested,
            'current_value': current_value,
            'gain_loss': gain,
            'gain_loss_pct': gain_pct,
            'cagr_pct': None,   # XIRR needs full txn history
            'purchase_date': str(as_of),
            'years_held': None,
            'tax_type': 'EEE (up to 60% exempt on withdrawal)',
            'notes': f'PRAN: {row["pran"]} | Units: {units} | NAV: ₹{nav} | As of: {as_of}'
        })

nps_df = pd.DataFrame(nps_records)
if not nps_df.empty:
    print(f'  NPS total invested : ₹{nps_df["invested_amount"].sum():,.0f}')
    print(f'  NPS current value  : ₹{nps_df["current_value"].sum():,.0f}')
print()

=== NPS ===
  ⚠️  nps_holdings.csv not found — skipping



In [9]:
# ── Cell 8: PPF ────────────────────────────────────────────────────────────────
print('=== PPF ===')

ppf = safe_read('ppf_holdings.csv')
ppf_records = []

if not ppf.empty:
    for _, row in ppf.iterrows():
        balance = float(row['current_balance'])
        annual_deposit = float(row['annual_deposit'])
        rate = float(row['interest_rate_pct'])
        opening = pd.to_datetime(row['opening_date']).date()
        maturity = pd.to_datetime(row['maturity_date']).date()
        years_left = max((maturity - TODAY).days / 365.25, 0)
        years_held = (TODAY - opening).days / 365.25
        projected = ppf_projected_balance(balance, annual_deposit, rate, int(years_left))

        # Rough total invested (no transaction history assumed)
        approx_invested = round(annual_deposit * years_held, 2)
        gain = round(balance - approx_invested, 2)

        ppf_records.append({
            'instrument_type': 'PPF',
            'name': f"{row['bank_name']} PPF — {row['account_number']}",
            'invested_amount': approx_invested,
            'current_value': balance,
            'gain_loss': gain,
            'gain_loss_pct': round((gain / approx_invested) * 100, 2) if approx_invested else 0,
            'cagr_pct': rate,
            'purchase_date': str(opening),
            'years_held': round(years_held, 2),
            'tax_type': 'EEE (fully exempt)',
            'notes': f'Rate: {rate}% | Matures: {maturity} | Projected at maturity: ₹{projected:,.0f}'
        })

ppf_df = pd.DataFrame(ppf_records)
if not ppf_df.empty:
    print(f'  PPF current balance : ₹{ppf_df["current_value"].sum():,.0f}')
print()

=== PPF ===
  ⚠️  ppf_holdings.csv not found — skipping



In [10]:
# ── Cell 9: GOLD ETF / SGB / BONDS ────────────────────────────────────────────
print('=== Gold ETF / Bonds ===')

gold = safe_read('gold_bonds_holdings.csv')
gold_records = []

gold_price_inr = gold_spot_price_inr()
if gold_price_inr:
    print(f'  Gold spot price: ₹{gold_price_inr:,.2f} / gram')

if not gold.empty:
    for _, row in gold.iterrows():
        itype = row['type']   # SGB / GOLD_ETF / GOVT_BOND / CORP_BOND
        units = float(row['units_or_grams'])
        issue_price = float(row['issue_price'])
        invested = round(units * issue_price, 2)
        purchase_date = pd.to_datetime(row['purchase_date']).date()
        years_held = (TODAY - purchase_date).days / 365.25

        # Current value
        if itype == 'SGB':
            # SGB value = gold price × grams (1 unit = 1 gram)
            current_price = gold_price_inr or issue_price
            current_value = round(units * current_price, 2)
            # Accrued coupon interest (2.5% p.a. on issue price, paid semi-annually)
            coupon_rate = float(row.get('interest_rate_pct', 2.5)) / 100
            accrued_coupon = round(invested * coupon_rate * years_held, 2)
            notes = f'SGB | Gold: ₹{current_price:,.0f}/g | Coupon accrued: ₹{accrued_coupon:,.0f}'
        elif itype == 'GOLD_ETF':
            ticker = row.get('ticker', '')
            try:
                current_price = round(yf.Ticker(ticker).history(period='1d')['Close'].iloc[-1], 2)
            except Exception:
                current_price = issue_price
            current_value = round(units * current_price, 2)
            notes = f'ETF: {ticker} | LTP: ₹{current_price}'
        else:
            # Bond: face value + coupon accrued (simplified)
            face_value = float(row.get('face_value', issue_price))
            coupon_rate = float(row.get('interest_rate_pct', 7.0)) / 100
            current_value = round(units * face_value, 2)
            accrued_coupon = round(invested * coupon_rate * years_held, 2)
            notes = f'{itype} | Face value: ₹{face_value} | Coupon accrued: ₹{accrued_coupon:,.0f}'

        gain = round(current_value - invested, 2)
        gain_pct = round((gain / invested) * 100, 2) if invested else 0
        cagr_val = cagr(invested, current_value, years_held)
        tax_type = 'LTCG (20% indexation)' if years_held >= 3 else 'STCG (slab)'

        gold_records.append({
            'instrument_type': itype,
            'name': row['instrument_name'],
            'invested_amount': invested,
            'current_value': current_value,
            'gain_loss': gain,
            'gain_loss_pct': gain_pct,
            'cagr_pct': cagr_val,
            'purchase_date': str(purchase_date),
            'years_held': round(years_held, 2),
            'tax_type': tax_type,
            'notes': notes
        })

gold_df = pd.DataFrame(gold_records)
if not gold_df.empty:
    print(f'  Gold/Bonds invested : ₹{gold_df["invested_amount"].sum():,.0f}')
    print(f'  Gold/Bonds value    : ₹{gold_df["current_value"].sum():,.0f}')
print()

=== Gold ETF / Bonds ===
  ⚠️  gold_bonds_holdings.csv not found — skipping
  Gold spot price: ₹14,251.81 / gram



In [11]:
# ── Cell 10: Merge into master portfolio ───────────────────────────────────────
print('=== Building master portfolio ===')

all_dfs = [df for df in [mf_df, eq_df, fd_df, nps_df, ppf_df, gold_df] if not df.empty]

if not all_dfs:
    print('⚠️  No data loaded yet — upload your CSVs to /kaggle/input/my-portfolio/ and re-run')
else:
    master = pd.concat(all_dfs, ignore_index=True)
    master['last_updated'] = str(TODAY)

    # Save
    out_path = os.path.join(OUTPUT_DIR, 'portfolio_master.csv')
    master.to_csv(out_path, index=False)
    print(f'✅ Saved: {out_path}  ({len(master)} holdings)')

=== Building master portfolio ===
⚠️  No data loaded yet — upload your CSVs to /kaggle/input/my-portfolio/ and re-run


In [12]:
# ── Cell 11: Portfolio summary ─────────────────────────────────────────────────
if all_dfs:
    total_invested = master['invested_amount'].sum()
    total_value = master['current_value'].sum()
    total_gain = total_value - total_invested
    total_gain_pct = (total_gain / total_invested) * 100 if total_invested else 0

    print('=' * 55)
    print('  PORTFOLIO SUMMARY')
    print('=' * 55)
    print(f'  Total invested    : ₹{total_invested:>15,.0f}')
    print(f'  Current value     : ₹{total_value:>15,.0f}')
    print(f'  Total gain / loss : ₹{total_gain:>15,.0f}  ({total_gain_pct:.1f}%)')
    print('=' * 55)

    by_type = master.groupby('instrument_type').agg(
        invested=('invested_amount', 'sum'),
        current=('current_value', 'sum')
    ).reset_index()
    by_type['gain'] = by_type['current'] - by_type['invested']
    by_type['gain_pct'] = (by_type['gain'] / by_type['invested'] * 100).round(1)
    by_type['allocation_pct'] = (by_type['current'] / total_value * 100).round(1)

    print()
    print(by_type[['instrument_type', 'invested', 'current', 'gain', 'gain_pct', 'allocation_pct']]
          .rename(columns={
              'instrument_type': 'Type',
              'invested': 'Invested (₹)',
              'current': 'Value (₹)',
              'gain': 'Gain (₹)',
              'gain_pct': 'Gain %',
              'allocation_pct': 'Alloc %'
          }).to_string(index=False))

    print()
    print('Top 5 holdings by current value:')
    print(master.nlargest(5, 'current_value')[['name', 'instrument_type', 'current_value', 'gain_loss_pct']]
          .rename(columns={
              'name': 'Holding',
              'instrument_type': 'Type',
              'current_value': 'Value (₹)',
              'gain_loss_pct': 'Gain %'
          }).to_string(index=False))

In [13]:
# ── Cell 12: Quick visualisation ──────────────────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

if all_dfs:
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    fig.suptitle('Portfolio Overview', fontsize=15, fontweight='bold')

    # Pie — allocation by type
    ax1 = axes[0]
    ax1.pie(
        by_type['current'],
        labels=by_type['instrument_type'],
        autopct='%1.1f%%',
        startangle=140,
        colors=plt.cm.Set2.colors[:len(by_type)]
    )
    ax1.set_title('Allocation by instrument')

    # Bar — invested vs current per type
    ax2 = axes[1]
    x = range(len(by_type))
    w = 0.35
    bars1 = ax2.bar([i - w/2 for i in x], by_type['invested'] / 1e5, w, label='Invested', color='#5c85d6')
    bars2 = ax2.bar([i + w/2 for i in x], by_type['current']  / 1e5, w, label='Current',  color='#57b894')
    ax2.set_xticks(list(x))
    ax2.set_xticklabels(by_type['instrument_type'], rotation=30, ha='right', fontsize=9)
    ax2.set_ylabel('Amount (₹ Lakhs)')
    ax2.set_title('Invested vs Current value')
    ax2.legend()
    ax2.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.1f L'))

    plt.tight_layout()
    chart_path = os.path.join(OUTPUT_DIR, 'portfolio_overview.png')
    plt.savefig(chart_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Chart saved: {chart_path}')

## ✅ Next steps

| Step | What to build next |
|------|-------------------|
| Notebook 2 | **OpenClaw agent** — RAG over `portfolio_master.csv` with tools: XIRR, tax estimator, goal projector |
| Notebook 3 | **Gradio UI** — natural language chat: *'Am I on track for ₹5Cr retirement?'* |
| Optional | Zerodha Kite API / Groww API integration to replace manual equity CSV |
| Optional | AMFI CAS PDF parser for auto-import of mutual fund statement |
